In [1]:
!pip install -q ucimlrepo

In [2]:
#Import libraries and create folders
import os
import json
import warnings
import zipfile

import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Create project folders
os.makedirs("/content/data/raw", exist_ok=True)
os.makedirs("/content/data/processed", exist_ok=True)
os.makedirs("/content/data/audit", exist_ok=True)

print("Project directories created.")

Project directories created.


In [4]:
#Download UCI Heart Disease Dataset
print("=" * 60)
print("DATA ACQUISITION")
print("=" * 60)

heart_disease = fetch_ucirepo(id=45)

X_raw = heart_disease.data.features.copy()
y_raw = heart_disease.data.targets.copy()

print("Dataset successfully acquired.")
print("Feature shape:", X_raw.shape)
print("Target shape:", y_raw.shape)

display(X_raw.head())
display(y_raw.head())

DATA ACQUISITION
Dataset successfully acquired.
Feature shape: (303, 13)
Target shape: (303, 1)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


,num
0,0
1,2
2,1
3,0
4,0


In [5]:
#Save the original raw dataset
raw_dataset = X_raw.copy()

# Add original UCI target
raw_dataset["num"] = y_raw["num"].values

raw_path = "/content/data/raw/heart_disease_raw.csv"

raw_dataset.to_csv(
    raw_path,
    index=False
)

print("Raw dataset saved to:")
print(raw_path)

print("\nRaw dataset shape:", raw_dataset.shape)

display(raw_dataset.head())

Raw dataset saved to:
/content/data/raw/heart_disease_raw.csv

Raw dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [6]:
#Save source information
source_info = {
    "dataset": "UCI Heart Disease",
    "UCI_dataset_id": 45,
    "source": "UCI Machine Learning Repository",
    "task": "Heart Disease Classification",
    "target_original": "num",
    "target_transformation": "num > 0 -> 1, num = 0 -> 0",
    "random_state": RANDOM_STATE
}

with open(
    "/content/data/audit/source_info.json",
    "w"
) as f:
    json.dump(source_info, f, indent=4)

print("Source information saved.")

Source information saved.


In [8]:
#Combine features and target
df = X_raw.copy()

df["num"] = y_raw["num"].values

print("Initial dataset shape:", df.shape)

display(df.head())

Initial dataset shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0,2
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0,1
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0,0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0,0


In [9]:
#Schema validation
print("=" * 60)
print("SCHEMA VALIDATION")
print("=" * 60)

expected_features = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal"
]

expected_columns = expected_features + ["num"]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

unexpected_columns = [
    col for col in df.columns
    if col not in expected_columns
]

if missing_columns:
    raise ValueError(
        f"Schema validation failed. Missing columns: {missing_columns}"
    )

if unexpected_columns:
    print("Warning: Unexpected columns:", unexpected_columns)

print("Schema validation PASSED.")
print("Expected columns found:", len(expected_columns))

SCHEMA VALIDATION
Schema validation PASSED.
Expected columns found: 14


In [10]:
#Data Type audit
dtype_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isnull().sum().values,
    "unique_values": df.nunique().values
})

display(dtype_audit)

dtype_audit.to_csv(
    "/content/data/audit/schema_audit.csv",
    index=False
)

print("Schema audit saved.")

,column,dtype,missing_count,unique_values
0,age,int64,0,41
1,sex,int64,0,2
2,cp,int64,0,4
3,trestbps,int64,0,50
4,chol,int64,0,152
5,fbs,int64,0,2
6,restecg,int64,0,3
7,thalach,int64,0,91
8,exang,int64,0,2
9,oldpeak,float64,0,40


Schema audit saved.


In [11]:
#Initial data quality audit
dtype_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isnull().sum().values,
    "unique_values": df.nunique().values
})

display(dtype_audit)

dtype_audit.to_csv(
    "/content/data/audit/schema_audit.csv",
    index=False
)

print("Schema audit saved.")

,column,dtype,missing_count,unique_values
0,age,int64,0,41
1,sex,int64,0,2
2,cp,int64,0,4
3,trestbps,int64,0,50
4,chol,int64,0,152
5,fbs,int64,0,2
6,restecg,int64,0,3
7,thalach,int64,0,91
8,exang,int64,0,2
9,oldpeak,float64,0,40


Schema audit saved.


In [12]:
#Duplicate check
duplicate_count = df.duplicated().sum()

print("Duplicate records:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)

    print(
        f"Removed {duplicate_count} duplicate records."
    )
else:
    print("No duplicate records found.")

Duplicate records: 0
No duplicate records found.


In [13]:
#Invalid value validation
invalid_counts = {}

invalid_counts["age"] = (
    df["age"] <= 0
).sum()

invalid_counts["trestbps"] = (
    df["trestbps"] <= 0
).sum()

invalid_counts["chol"] = (
    df["chol"] <= 0
).sum()

invalid_counts["thalach"] = (
    df["thalach"] <= 0
).sum()

invalid_counts["num"] = (
    ~df["num"].isin([0, 1, 2, 3, 4])
).sum()

invalid_audit = pd.DataFrame(
    list(invalid_counts.items()),
    columns=[
        "column",
        "invalid_count"
    ]
)

display(invalid_audit)

invalid_audit.to_csv(
    "/content/data/audit/invalid_records.csv",
    index=False
)

print("Invalid-record audit saved.")

,column,invalid_count
0,age,0
1,trestbps,0
2,chol,0
3,thalach,0
4,num,0


Invalid-record audit saved.


In [14]:
#Remove invalid records
before_invalid_removal = len(df)

df = df[
    (df["age"] > 0) &
    (df["trestbps"] > 0) &
    (df["chol"] > 0) &
    (df["thalach"] > 0) &
    (df["num"].isin([0, 1, 2, 3, 4]))
].copy()

after_invalid_removal = len(df)

print(
    "Invalid records removed:",
    before_invalid_removal - after_invalid_removal
)

print("Remaining records:", len(df))

Invalid records removed: 0
Remaining records: 303


In [16]:
#Convert target into binary classification
df["target"] = (
    df["num"] > 0
).astype(int)

df.drop(
    columns=["num"],
    inplace=True
)

print("Binary target distribution:")
display(
    df["target"].value_counts()
)

print("\nTarget percentages:")
display(
    df["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Binary target distribution:


,count
target,
0,164
1,139



Target percentages:


,proportion
target,
0,54.13
1,45.87


In [17]:
#Seperate X and Y
X = df.drop(
    columns=["target"]
)

y = df["target"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (303, 13)
Target shape: (303,)


In [18]:
#Train/ Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining target distribution:")
display(
    y_train.value_counts(normalize=True)
)

print("\nTesting target distribution:")
display(
    y_test.value_counts(normalize=True)
)

Training samples: 242
Testing samples: 61

Training target distribution:


,proportion
target,
0,0.541322
1,0.458678



Testing target distribution:


,proportion
target,
0,0.540984
1,0.459016


In [19]:
#Outlier report
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

outlier_report = []

for column in numeric_features:

    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    train_outliers = (
        (X_train[column] < lower) |
        (X_train[column] > upper)
    ).sum()

    test_outliers = (
        (X_test[column] < lower) |
        (X_test[column] > upper)
    ).sum()

    outlier_report.append({
        "feature": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower,
        "upper_bound": upper,
        "train_outliers": train_outliers,
        "test_outliers": test_outliers
    })

outlier_report = pd.DataFrame(
    outlier_report
)

display(outlier_report)

outlier_report.to_csv(
    "/content/data/audit/outlier_report.csv",
    index=False
)

,feature,Q1,Q3,IQR,lower_bound,upper_bound,train_outliers,test_outliers
0,age,48.00,61.00,13.00,28.500,80.500,0,0
1,sex,0.00,1.00,1.00,-1.500,2.500,0,0
2,cp,2.25,4.00,1.75,-0.375,6.625,0,0
3,trestbps,120.00,140.00,20.00,90.000,170.000,6,3
4,chol,212.00,277.75,65.75,113.375,376.375,5,0
5,fbs,0.00,0.00,0.00,0.000,0.000,35,10
6,restecg,0.00,2.00,2.00,-3.000,5.000,0,0
7,thalach,134.50,166.00,31.50,87.250,213.250,1,0
8,exang,0.00,1.00,1.00,-1.500,2.500,0,0
9,oldpeak,0.00,1.60,1.60,-2.400,4.000,4,1


In [20]:
#Create IQR clipping transformer
class IQRClipper(
    BaseEstimator,
    TransformerMixin
):

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):

        X_df = pd.DataFrame(X)

        Q1 = X_df.quantile(0.25)
        Q3 = X_df.quantile(0.75)

        IQR = Q3 - Q1

        self.lower_bounds_ = (
            Q1 - self.factor * IQR
        )

        self.upper_bounds_ = (
            Q3 + self.factor * IQR
        )

        return self

    def transform(self, X):

        X_df = pd.DataFrame(X).copy()

        for column in X_df.columns:

            X_df[column] = X_df[column].clip(
                lower=self.lower_bounds_[column],
                upper=self.upper_bounds_[column]
            )

        return X_df.values

In [21]:
#Build leakage free -preprocessing pipeline
preprocessing_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "outlier_clipper",
        IQRClipper(
            factor=1.5
        )
    ),

    (
        "scaler",
        StandardScaler()
    )
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [22]:
#Fit only on training data
X_train_processed = (
    preprocessing_pipeline
    .fit_transform(X_train)
)

print("Training preprocessing complete.")

Training preprocessing complete.


In [24]:
#Transform test data
X_test_processed = (
    preprocessing_pipeline
    .transform(X_test)
)

print("Testing preprocessing complete.")

Testing preprocessing complete.


In [25]:
#Convert processed arrays to DataFrames
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=X_train.columns,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=X_test.columns,
    index=X_test.index
)

display(X_train_processed.head())

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
180,-0.729485,0.68313,0.870169,-0.398560,0.531065,0.0,1.022996,0.712047,-0.696177,-0.455191,0.675060,-0.712457,1.179973
208,0.050166,0.68313,-1.184278,-0.037318,0.280548,0.0,-0.981579,0.221597,-0.696177,-0.927560,-0.958585,-0.712457,-0.878070
167,-0.061212,-1.46385,-1.184278,0.083095,0.823334,0.0,1.022996,0.399942,1.436416,-0.927560,-0.958585,0.510342,-0.878070
105,-0.061212,0.68313,-1.184278,-1.361870,1.261738,0.0,-0.981579,0.266183,-0.696177,-0.927560,-0.958585,-0.712457,1.179973
297,0.272924,-1.46385,0.870169,0.564751,-0.157856,0.0,-0.981579,-1.205170,1.436416,-0.738613,0.675060,-0.712457,1.179973


In [26]:
#Post-processing quality check
train_missing = (
    X_train_processed
    .isnull()
    .sum()
    .sum()
)

test_missing = (
    X_test_processed
    .isnull()
    .sum()
    .sum()
)

print(
    "Training missing values:",
    train_missing
)

print(
    "Testing missing values:",
    test_missing
)

assert train_missing == 0
assert test_missing == 0

print("Missing-value validation PASSED.")

Training missing values: 0
Testing missing values: 0
Missing-value validation PASSED.


In [27]:
#Data drift / PSI
def calculate_psi(
    expected,
    actual,
    bins=10
):

    expected = np.asarray(expected)
    actual = np.asarray(actual)

    breakpoints = np.percentile(
        expected,
        np.linspace(
            0,
            100,
            bins + 1
        )
    )

    breakpoints = np.unique(
        breakpoints
    )

    if len(breakpoints) < 3:
        return 0.0

    expected_counts = np.histogram(
        expected,
        bins=breakpoints
    )[0]

    actual_counts = np.histogram(
        actual,
        bins=breakpoints
    )[0]

    expected_pct = (
        expected_counts /
        len(expected)
    )

    actual_pct = (
        actual_counts /
        len(actual)
    )

    expected_pct = np.where(
        expected_pct == 0,
        0.0001,
        expected_pct
    )

    actual_pct = np.where(
        actual_pct == 0,
        0.0001,
        actual_pct
    )

    psi = np.sum(
        (actual_pct - expected_pct)
        *
        np.log(
            actual_pct /
            expected_pct
        )
    )

    return psi

In [28]:
#Generate drift report
drift_results = []

for column in X_train.columns:

    psi_value = calculate_psi(
        X_train[column].dropna(),
        X_test[column].dropna()
    )

    if psi_value < 0.10:
        status = "Low Drift"

    elif psi_value < 0.25:
        status = "Moderate Drift"

    else:
        status = "Significant Drift"

    drift_results.append({
        "feature": column,
        "PSI": round(
            psi_value,
            4
        ),
        "drift_status": status
    })

drift_report = pd.DataFrame(
    drift_results
)

display(drift_report)

drift_report.to_csv(
    "/content/data/audit/data_drift_report.csv",
    index=False
)

print("Drift report saved.")

,feature,PSI,drift_status
0,age,0.5365,Significant Drift
1,sex,0.0000,Low Drift
2,cp,0.0182,Low Drift
3,trestbps,0.2998,Significant Drift
4,chol,0.2776,Significant Drift
5,fbs,0.0000,Low Drift
6,restecg,0.0000,Low Drift
7,thalach,0.1028,Moderate Drift
8,exang,0.0000,Low Drift
9,oldpeak,0.1935,Moderate Drift


Drift report saved.


In [29]:
#Save processed training matrix
train_matrix = X_train_processed.copy()

train_matrix["target"] = y_train.values

train_matrix.to_csv(
    "/content/data/processed/train_features.csv",
    index=False
)

print("Training matrix saved.")
print(train_matrix.shape)

Training matrix saved.
(242, 14)


In [30]:
#Save processed testing matrix
test_matrix = X_test_processed.copy()

test_matrix["target"] = y_test.values

test_matrix.to_csv(
    "/content/data/processed/test_features.csv",
    index=False
)

print("Testing matrix saved.")
print(test_matrix.shape)

Testing matrix saved.
(61, 14)


In [31]:
#Save y_train and y_test
y_train.to_csv(
    "/content/data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "/content/data/processed/y_test.csv",
    index=False
)

print("y_train.csv saved.")
print("y_test.csv saved.")

y_train.csv saved.
y_test.csv saved.


In [32]:
#Final quality audit
final_audit = []

for column in X_train_processed.columns:

    final_audit.append({

        "feature": column,

        "train_missing":
            int(
                X_train_processed[column]
                .isnull()
                .sum()
            ),

        "test_missing":
            int(
                X_test_processed[column]
                .isnull()
                .sum()
            ),

        "train_mean":
            round(
                X_train_processed[column]
                .mean(),
                4
            ),

        "train_std":
            round(
                X_train_processed[column]
                .std(),
                4
            ),

        "test_mean":
            round(
                X_test_processed[column]
                .mean(),
                4
            ),

        "test_std":
            round(
                X_test_processed[column]
                .std(),
                4
            ),

        "data_type":
            str(
                X_train_processed[column]
                .dtype
            )
    })

final_quality_audit = pd.DataFrame(
    final_audit
)

display(final_quality_audit)

final_quality_audit.to_csv(
    "/content/data/audit/final_data_quality_audit.csv",
    index=False
)

print("Final quality audit saved.")

,feature,train_missing,test_missing,train_mean,train_std,test_mean,test_std,data_type
0,age,0,0,-0.0,1.0021,-0.0612,1.0319,float64
1,sex,0,0,0.0,1.0021,-0.0208,1.0162,float64
2,cp,0,0,0.0,1.0021,0.0282,0.9285,float64
3,trestbps,0,0,0.0,1.0021,0.2173,0.9927,float64
4,chol,0,0,-0.0,1.0021,-0.2995,0.9516,float64
5,fbs,0,0,0.0,0.0000,0.0000,0.0000,float64
6,restecg,0,0,0.0,1.0021,0.0536,0.9849,float64
7,thalach,0,0,-0.0,1.0021,-0.0817,1.0588,float64
8,exang,0,0,0.0,1.0021,0.0030,1.0094,float64
9,oldpeak,0,0,0.0,1.0021,0.1999,1.2127,float64


Final quality audit saved.


In [33]:
#Verify all files
print("=" * 60)
print("GENERATED FILES")
print("=" * 60)

for root, dirs, files in os.walk("/content/data"):

    for file in files:

        path = os.path.join(
            root,
            file
        )

        size_kb = os.path.getsize(path) / 1024

        print(
            f"{path}  |  {size_kb:.2f} KB"
        )

GENERATED FILES
/content/data/raw/heart_disease_raw.csv  |  12.18 KB
/content/data/audit/final_data_quality_audit.csv  |  0.62 KB
/content/data/audit/invalid_records.csv  |  0.06 KB
/content/data/audit/data_drift_report.csv  |  0.33 KB
/content/data/audit/source_info.json  |  0.26 KB
/content/data/audit/outlier_report.csv  |  0.51 KB
/content/data/audit/schema_audit.csv  |  0.26 KB
/content/data/processed/train_features.csv  |  56.86 KB
/content/data/processed/test_features.csv  |  14.40 KB
/content/data/processed/y_train.csv  |  0.48 KB
/content/data/processed/y_test.csv  |  0.13 KB


In [34]:
#Create ZIP for GitHub
zip_path = "/content/DE_data_files.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(
        "/content/data"
    ):

        for file in files:

            file_path = os.path.join(
                root,
                file
            )

            arcname = os.path.relpath(
                file_path,
                "/content"
            )

            zipf.write(
                file_path,
                arcname
            )

print("ZIP created:")
print(zip_path)

ZIP created:
/content/DE_data_files.zip


In [35]:
#Download the ZIP
from google.colab import files

files.download(
    "/content/DE_data_files.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>